##### The following practice code is intended for educational purposes only. For contact :  audit@korea.ac.kr, Sungryel Lim Ph.D

##### This practice code is not a completed commercial version but has been developed for educational purposes.

# 00. Knowledge Asset 워밍업 실습
## BGE-M3 + FAISS + Qwen2.5-1.5B

이 Notebook은 본격적인 LoRA / Fine-Tuning 실습 전에 진행하는 **개념 이해용 워밍업**입니다.

이번 실습에서는 회사 내부의 작은 Knowledge Asset을 여러 문서로 나누고,
각 문서를 Embedding한 뒤 FAISS에 저장하여
사용자 질문과 가장 관련 있는 Knowledge를 검색합니다.

검색된 Knowledge만 기존 Qwen2.5-1.5B-Instruct에 Context로 전달하여 답변을 생성합니다.

이번 실습의 핵심 흐름은 다음과 같습니다.

```text
Knowledge Assets
      ↓
BGE-M3 Embedding
      ↓
FAISS Index
      ↓
사용자 질문
      ↓
질문 Embedding
      ↓
Semantic Search
      ↓
Top-K Knowledge
      ↓
Qwen2.5-1.5B
      ↓
답변
```

이번 Notebook에서는 여기까지만 다룹니다.

- Embedding
- FAISS
- Top-K Semantic Search
- 검색된 Knowledge를 Qwen Context로 전달

다음과 같은 본격적인 RAG 구성 요소는 이번 범위에서 다루지 않습니다.

- 복잡한 Chunking 전략
- Metadata Filtering
- Hybrid Search
- Reranking
- 외부 Vector DB
- API 기반 RAG Pipeline

즉, **Knowledge Asset을 검색 가능한 형태로 만들고 sLLM과 결합하는 최소 구조를 체험하는 것**이 목적입니다.


## 1. 현재 실행 환경과 Hugging Face 캐시 확인

이후 실습과 동일한 모델을 사용합니다.

- 생성 모델: `Qwen/Qwen2.5-1.5B-Instruct`
- Embedding 모델: `BAAI/bge-m3`

이미 다운로드된 모델이 있다면 Hugging Face 캐시를 재사용합니다.


In [1]:
import os
from pathlib import Path

from huggingface_hub.constants import HF_HUB_CACHE

print("현재 작업 디렉토리(CWD)")
print(" →", os.getcwd())

print("\nHugging Face Hub 캐시 경로")
print(" →", HF_HUB_CACHE)

print("\n캐시 디렉토리 존재 여부")
print(" →", Path(HF_HUB_CACHE).exists())


현재 작업 디렉토리(CWD)
 → /Users/seohyeokin/workspace/skala-sLLM/sllm-main-std

Hugging Face Hub 캐시 경로
 → /Users/seohyeokin/.cache/huggingface/hub

캐시 디렉토리 존재 여부
 → True


## 2. 필요한 라이브러리 불러오기

이번 실습에서는 다음 라이브러리를 사용합니다.

- `torch`
  - Qwen 실행 장치(CUDA / MPS / CPU) 제어

- `transformers`
  - Qwen2.5-1.5B-Instruct 로드

- `sentence_transformers`
  - BGE-M3 Embedding 모델 로드

- `faiss`
  - Embedding Vector를 저장하고 유사도 검색

> 기존 `requirements.txt` 환경에 `sentence-transformers`, `faiss-cpu`가 포함되어 있다고 가정합니다.


In [2]:
import numpy as np
import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


## 3. 실행 장치 결정

Qwen 생성 모델은 가능한 경우 GPU를 사용합니다.

우선순위는 다음과 같습니다.

```text
CUDA
 ↓
MPS
 ↓
CPU
```

BGE-M3 Embedding은 이번 워밍업에서는 안정성을 위해 CPU에서 실행합니다.
문서 수가 매우 적기 때문에 속도 차이는 크지 않습니다.


In [3]:
# Qwen 생성 모델에서 사용할 장치를 선택합니다.

if torch.cuda.is_available():
    device = torch.device("cuda:0")

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")

else:
    device = torch.device("cpu")


# Qwen 모델의 dtype을 결정합니다.
#
# GPU에서는 FP16,
# CPU에서는 FP32를 사용합니다.
if device.type in {"cuda", "mps"}:
    dtype = torch.float16

else:
    dtype = torch.float32


print("Qwen 실행 장치 :", device)
print("Qwen dtype     :", dtype)
print("Embedding 장치 : cpu")


Qwen 실행 장치 : mps
Qwen dtype     : torch.float16
Embedding 장치 : cpu


## 4. Qwen Base Model 로드

Fine-Tuning 이전의 원본 Qwen 모델을 사용합니다.

이미 Hugging Face 캐시에 존재하면 다시 다운로드하지 않고 재사용합니다.


In [4]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


# Tokenizer는 문자열과 Token ID 사이를 변환합니다.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)


# PAD Token이 없다면 EOS Token을 사용합니다.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Fine-Tuning 이전 Base Qwen을 로드합니다.
model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME,
        dtype=dtype,
        low_cpu_mem_usage=True,
    )
    .to(device)
)


# 이번 Notebook은 학습이 아니라 추론만 수행합니다.
model.eval()


print("Base Qwen 로드 완료")
print("Model :", MODEL_NAME)
print("Device:", device)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base Qwen 로드 완료
Model : Qwen/Qwen2.5-1.5B-Instruct
Device: mps


## 5. BGE-M3 Embedding 모델 로드

Embedding 모델은 문장을 바로 답변하는 모델이 아닙니다.

문장의 의미를 숫자 Vector로 변환하는 역할을 합니다.

예를 들어 다음 두 문장은 표현은 다르지만 의미가 비슷합니다.

```text
AI 교육비는 얼마까지 지원되나요?
연간 교육비 지원 한도가 어떻게 되나요?
```

Embedding 공간에서는 이런 의미가 비슷한 문장들이 서로 가까운 Vector가 되도록 학습되어 있습니다.


In [5]:
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"


# BGE-M3를 CPU에 로드합니다.
#
# 이번 예제는 Knowledge 문서가 적기 때문에
# CPU로도 충분히 빠르게 실행할 수 있습니다.
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu",
)


print("Embedding Model 로드 완료")
print("Model :", EMBEDDING_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding Model 로드 완료
Model : BAAI/bge-m3


## 6. 회사 Knowledge Asset 정의

이번에는 하나의 긴 문자열을 Qwen에게 그대로 전달하지 않습니다.

회사가 가지고 있는 여러 내부 문서를 각각 별도의 Knowledge Document로 관리한다고 가정합니다.

각 문서는 다음 정보를 가집니다.

- `id`
- `title`
- `content`

실제 기업에서는 이런 문서가 사내 규정, 매뉴얼, 제품 설명서, FAQ 등이 될 수 있습니다.


In [6]:
# 가상의 Human AI Corporation 내부 Knowledge Asset

knowledge_documents = [
    {
        "id": "HR-EDU-001",
        "title": "AI 교육비 지원 한도",
        "content": (
            "Human AI Corporation의 AI 교육비 지원 제도 코드는 "
            "HAI-EDU-2026이다. 임직원 1인당 연간 최대 지원 금액은 "
            "120만원이다."
        ),
    },
    {
        "id": "HR-EDU-002",
        "title": "AI 교육비 신청 기한",
        "content": (
            "AI 교육비 지원 신청은 교육 시작일 기준 최소 14일 전에 "
            "완료해야 한다."
        ),
    },
    {
        "id": "HR-EDU-003",
        "title": "AI 교육비 승인 절차",
        "content": (
            "AI 교육비 승인 절차는 팀장 승인, HR 검토, 최종 승인 "
            "순서로 진행된다."
        ),
    },
    {
        "id": "HR-EDU-004",
        "title": "AI 교육비 지원 제외 조건",
        "content": (
            "교육 수료율이 80% 미만이면 교육비 지원 대상에서 제외된다. "
            "사전 승인을 받지 않은 교육은 원칙적으로 소급 지원하지 않는다."
        ),
    },
    {
        "id": "HR-LEAVE-001",
        "title": "리프레시 휴가",
        "content": (
            "입사 후 3년을 충족한 직원에게 리프레시 휴가 5일을 부여한다."
        ),
    },
    {
        "id": "HR-WORK-001",
        "title": "재택근무",
        "content": (
            "재택근무는 주 최대 2회까지 가능하며 직속 팀장의 사전 승인이 필요하다."
        ),
    },
]


print(f"Knowledge Document 수: {len(knowledge_documents)}")

for doc in knowledge_documents:
    print(
        f"- {doc['id']} | "
        f"{doc['title']}"
    )


Knowledge Document 수: 6
- HR-EDU-001 | AI 교육비 지원 한도
- HR-EDU-002 | AI 교육비 신청 기한
- HR-EDU-003 | AI 교육비 승인 절차
- HR-EDU-004 | AI 교육비 지원 제외 조건
- HR-LEAVE-001 | 리프레시 휴가
- HR-WORK-001 | 재택근무


## 7. Knowledge Document를 Embedding하기

각 문서의 `title + content`를 하나의 문자열로 만든 뒤 BGE-M3로 Embedding합니다.

Embedding 결과는 사람이 직접 읽는 문장이 아니라 숫자 Vector입니다.

```text
문서
  ↓
BGE-M3
  ↓
[0.012, -0.034, 0.081, ...]
```

이 Vector를 이용하면 질문과 문서 사이의 의미적 유사도를 계산할 수 있습니다.


In [7]:
# 검색 시 제목 정보도 의미 판단에 도움이 되도록
# title과 content를 함께 Embedding합니다.
knowledge_texts = [
    f"{doc['title']}\n{doc['content']}"
    for doc in knowledge_documents
]


# normalize_embeddings=True
# → 각 Vector의 길이를 1로 정규화합니다.
#
# 이렇게 하면 Inner Product를 이용해
# Cosine Similarity와 같은 방식으로 비교할 수 있습니다.
knowledge_embeddings = embedding_model.encode(
    knowledge_texts,
    normalize_embeddings=True,
)


# FAISS에서 일반적으로 float32 Vector를 사용하므로
# 타입을 명시적으로 맞춥니다.
knowledge_embeddings = np.asarray(
    knowledge_embeddings,
    dtype="float32",
)


print("Embedding 완료")
print("문서 수        :", knowledge_embeddings.shape[0])
print("Embedding 차원 :", knowledge_embeddings.shape[1])


Embedding 완료
문서 수        : 6
Embedding 차원 : 1024


## 8. FAISS Index 구축

이제 Embedding Vector를 FAISS Index에 저장합니다.

이번 예제에서는 `IndexFlatIP`를 사용합니다.

- `IP`
  - Inner Product

- 앞에서 Vector를 정규화했기 때문에
  - Inner Product 값은 Cosine Similarity와 같은 방식으로 해석할 수 있습니다.

문서가 매우 적기 때문에 복잡한 Index 구조는 필요하지 않습니다.


In [8]:
# Embedding Vector의 차원 수를 가져옵니다.
embedding_dimension = knowledge_embeddings.shape[1]


# IndexFlatIP
# → 모든 Vector와 직접 Inner Product를 계산하는
#   가장 단순한 FAISS Index입니다.
#
# 작은 교육용 Knowledge Base에는 충분합니다.
faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)


# Knowledge Embedding을 FAISS Index에 추가합니다.
faiss_index.add(
    knowledge_embeddings
)


print("FAISS Index 구축 완료")
print("저장된 Vector 수:", faiss_index.ntotal)


FAISS Index 구축 완료
저장된 Vector 수: 6


## 9. Semantic Search 함수 만들기

이제 사용자 질문이 들어오면 다음 순서로 검색합니다.

```text
질문
  ↓
BGE-M3 Embedding
  ↓
FAISS 검색
  ↓
Top-K 문서 반환
```

이 함수가 이번 Mini Knowledge Base의 핵심입니다.


In [9]:
def search_knowledge(
    question: str,
    top_k: int = 2,
) -> list[dict]:
    # 사용자 질문을 BGE-M3로 Embedding합니다.
    #
    # Knowledge Document와 동일하게
    # normalize_embeddings=True를 사용합니다.
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True,
    )

    # FAISS 검색을 위해 float32로 변환합니다.
    query_embedding = np.asarray(
        query_embedding,
        dtype="float32",
    )

    # FAISS에서 가장 유사한 Top-K Vector를 검색합니다.
    #
    # scores
    # → 질문과 검색된 문서의 유사도 점수
    #
    # indices
    # → 검색된 Knowledge Document의 위치
    scores, indices = faiss_index.search(
        query_embedding,
        top_k,
    )

    results = []

    # 검색된 Index를 실제 Knowledge Document와 연결합니다.
    for score, doc_index in zip(
        scores[0],
        indices[0],
    ):
        doc = knowledge_documents[
            int(doc_index)
        ]

        results.append(
            {
                "id": doc["id"],
                "title": doc["title"],
                "content": doc["content"],
                "score": float(score),
            }
        )

    return results


## 10. 실제 Semantic Search 실행

먼저 Qwen에게 답변을 생성시키지 않고,
FAISS가 어떤 Knowledge를 검색하는지만 확인합니다.

질문:

```text
AI 교육비는 연간 얼마까지 지원받을 수 있나요?
```

문장 자체가 Knowledge Document와 완전히 동일하지 않아도
의미가 비슷하면 관련 문서를 찾을 수 있는지 확인해 봅니다.


In [10]:
question = (
    "AI 교육비는 연간 얼마까지 "
    "지원받을 수 있나요?"
)


search_results = search_knowledge(
    question,
    top_k=2,
)


print("[Question]")
print(question)


for rank, result in enumerate(
    search_results,
    start=1,
):
    print(
        f"\n[TOP {rank}] "
        f"score={result['score']:.4f}"
    )

    print(
        "ID     :",
        result["id"],
    )

    print(
        "Title  :",
        result["title"],
    )

    print(
        "Content:",
        result["content"],
    )


[Question]
AI 교육비는 연간 얼마까지 지원받을 수 있나요?

[TOP 1] score=0.7812
ID     : HR-EDU-001
Title  : AI 교육비 지원 한도
Content: Human AI Corporation의 AI 교육비 지원 제도 코드는 HAI-EDU-2026이다. 임직원 1인당 연간 최대 지원 금액은 120만원이다.

[TOP 2] score=0.6470
ID     : HR-EDU-004
Title  : AI 교육비 지원 제외 조건
Content: 교육 수료율이 80% 미만이면 교육비 지원 대상에서 제외된다. 사전 승인을 받지 않은 교육은 원칙적으로 소급 지원하지 않는다.


## 11. Qwen 답변 생성 함수

이제 검색된 Knowledge를 Qwen에게 전달하여 답변을 생성합니다.

중요한 점은 Qwen에게 모든 Knowledge Document를 넣는 것이 아니라,
**FAISS가 검색한 Top-K Knowledge만 Context로 전달한다는 것**입니다.


In [11]:
def generate_answer(
    question: str,
    retrieved_documents: list[dict],
) -> str:
    # 검색된 Knowledge Document를
    # 하나의 Context 문자열로 합칩니다.
    context = "\n\n".join(
        (
            f"[{doc['id']}] {doc['title']}\n"
            f"{doc['content']}"
        )
        for doc in retrieved_documents
    )

    # 모델에게 주어지는 System Prompt입니다.
    #
    # 검색된 Knowledge에 없는 내용은
    # 임의로 만들지 않도록 명시합니다.
    system_prompt = f'''
당신은 Human AI Corporation의 사내 규정 안내 AI Assistant입니다.

반드시 아래 [Retrieved Knowledge]에 명시된 내용만을 근거로 답변하세요.

[답변 원칙]
- Retrieved Knowledge에 직접 명시된 사실만 답변에 사용하세요.
- Retrieved Knowledge의 내용을 바탕으로 새로운 회사 규정을 추론하지 마세요.
- 일반적인 기업 관행이나 외부 지식을 사용하지 마세요.
- 질문에 대한 답이 Retrieved Knowledge에 직접 명시되어 있지 않다면
  반드시 "현재 Knowledge Base에서 해당 내용을 확인할 수 없습니다."라고 답변하세요.
- 정보가 없다는 이유로 반대 사실을 단정하지 마세요.
  예를 들어 "지원한다"는 정보가 없다고 해서 "지원하지 않는다"고 답변하면 안 됩니다.

[Retrieved Knowledge]
{context}
'''.strip()

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    # Qwen Chat Template 적용
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 문자열을 Token ID로 변환
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(device)

    # 이번 실습은 추론이므로
    # Gradient 계산을 비활성화합니다.
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,

            # Base / 검색 결과 비교가 쉽도록
            # Sampling은 사용하지 않습니다.
            do_sample=False,

            use_cache=True,

            pad_token_id=(
                tokenizer.pad_token_id
            ),

            eos_token_id=(
                tokenizer.eos_token_id
            ),
        )

    # 입력 Prompt 부분을 제외하고
    # 모델이 새롭게 생성한 Token만 가져옵니다.
    input_length = (
        inputs["input_ids"].shape[1]
    )

    generated_tokens = (
        outputs[0][input_length:]
    )

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return answer.strip()


## 12. Knowledge 검색 + Qwen 답변

이제 Semantic Search와 Qwen을 연결합니다.

```text
질문
  ↓
BGE-M3
  ↓
FAISS
  ↓
Top-K Knowledge
  ↓
Qwen
  ↓
답변
```


In [12]:
search_results = search_knowledge(
    question,
    top_k=2,
)


answer = generate_answer(
    question=question,
    retrieved_documents=search_results,
)


print("[Question]")
print(question)

print("\n[Retrieved Knowledge]")

for result in search_results:
    print(
        f"- {result['id']} | "
        f"{result['title']} | "
        f"score={result['score']:.4f}"
    )

print("\n[Qwen Answer]")
print(answer)


[Question]
AI 교육비는 연간 얼마까지 지원받을 수 있나요?

[Retrieved Knowledge]
- HR-EDU-001 | AI 교육비 지원 한도 | score=0.7812
- HR-EDU-004 | AI 교육비 지원 제외 조건 | score=0.6470

[Qwen Answer]
AI 교육비는 연간 120만원까지 지원받을 수 있습니다.


## 13. 여러 질문으로 Knowledge Base 확인

이번에는 서로 다른 주제의 질문을 넣어
FAISS가 질문마다 다른 Knowledge를 검색하는지 확인합니다.

이 부분이 단순히 Knowledge 전체를 Qwen Prompt에 넣는 방식과 다른 점입니다.


In [13]:
test_questions = [
    "AI 교육비 신청은 교육 시작 며칠 전까지 해야 하나요?",
    "AI 교육비는 어떤 승인 절차를 거치나요?",
    "수료율이 70%라면 교육비 지원을 받을 수 있나요?",
    "재택근무는 일주일에 몇 번 가능한가요?",
    "리프레시 휴가는 언제 받을 수 있나요?",
]


for index, test_question in enumerate(
    test_questions,
    start=1,
):
    retrieved = search_knowledge(
        test_question,
        top_k=2,
    )

    answer = generate_answer(
        question=test_question,
        retrieved_documents=retrieved,
    )

    print("\n" + "=" * 70)

    print(
        f"[Question {index}]"
    )

    print(
        test_question
    )

    print("\n[Retrieved]")

    for result in retrieved:
        print(
            f"- {result['id']} | "
            f"{result['title']} | "
            f"{result['score']:.4f}"
        )

    print("\n[Answer]")
    print(answer)

print("\n" + "=" * 70)



[Question 1]
AI 교육비 신청은 교육 시작 며칠 전까지 해야 하나요?

[Retrieved]
- HR-EDU-002 | AI 교육비 신청 기한 | 0.8096
- HR-EDU-003 | AI 교육비 승인 절차 | 0.6302

[Answer]
AI 교육비 신청은 교육 시작 일 기준 최소 14일 전에 완료해야 합니다.

[Question 2]
AI 교육비는 어떤 승인 절차를 거치나요?

[Retrieved]
- HR-EDU-003 | AI 교육비 승인 절차 | 0.7915
- HR-EDU-004 | AI 교육비 지원 제외 조건 | 0.6485

[Answer]
AI 교육비 승인 절차는 다음과 같습니다:

1. **팀장 승인**: AI 교육 프로그램에 참여하는 직원들의 팀장이 승인이 필요합니다.
2. **HR 검토**: AI 교육 비용을 포함한 전체 예산 계획과 관련 문서를 HR에게 검토받습니다.
3. **최종 승인**: HR가 승인 후, 최종적으로 교육비 승인을 받습니다.

이러한 절차를 통해 AI 교육비가 정확하게 처리되고, 필요한 경우 교육비 지원을 위한 추가적인 서류와 문서를 준비해야 합니다.

[Question 3]
수료율이 70%라면 교육비 지원을 받을 수 있나요?

[Retrieved]
- HR-EDU-004 | AI 교육비 지원 제외 조건 | 0.6913
- HR-EDU-001 | AI 교육비 지원 한도 | 0.4828

[Answer]
네, 수료율이 70%라면 교육비 지원을 받을 수 있습니다. 그러나 이 경우 지원금액은 120만원보다 적게 지급될 것입니다.

[Question 4]
재택근무는 일주일에 몇 번 가능한가요?

[Retrieved]
- HR-WORK-001 | 재택근무 | 0.7673
- HR-LEAVE-001 | 리프레시 휴가 | 0.4619

[Answer]
재택근무는 주 최대 2회까지 가능합니다.

[Question 5]
리프레시 휴가는 언제 받을 수 있나요?

[Retrieved]
- HR-LEAVE-001 | 리프레

## 14. Knowledge Base에 없는 질문

이번에는 Knowledge Base에 존재하지 않는 정책을 질문합니다.

예:

```text
해외 AI 컨퍼런스 항공료를 전액 지원하나요?
```

FAISS는 항상 가장 가까운 문서를 반환하기 때문에,
검색 결과가 존재한다고 해서 그 문서가 반드시 정답이라는 뜻은 아닙니다.

이 점은 실제 RAG 시스템에서 매우 중요합니다.

실제 시스템에서는 다음과 같은 추가 장치를 사용할 수 있습니다.

- 유사도 Threshold
- Metadata Filtering
- Reranker
- "검색 결과 없음" 처리
- Hallucination Guard

이번 실습에서는 개념만 확인합니다.


In [14]:
unknown_question = (
    "Human AI Corporation은 해외 AI 컨퍼런스 참석 시 "
    "항공료를 전액 지원하나요?"
)


unknown_results = search_knowledge(
    unknown_question,
    top_k=2,
)


print("[Unknown Question]")
print(unknown_question)

print("\n[Retrieved Knowledge]")

for result in unknown_results:
    print(
        f"- {result['id']} | "
        f"{result['title']} | "
        f"score={result['score']:.4f}"
    )


unknown_answer = generate_answer(
    question=unknown_question,
    retrieved_documents=unknown_results,
)


print("\n[Qwen Answer]")
print(unknown_answer)


[Unknown Question]
Human AI Corporation은 해외 AI 컨퍼런스 참석 시 항공료를 전액 지원하나요?

[Retrieved Knowledge]
- HR-EDU-001 | AI 교육비 지원 한도 | score=0.5647
- HR-EDU-003 | AI 교육비 승인 절차 | score=0.4470

[Qwen Answer]
현재 Knowledge Base에서 해당 내용을 확인할 수 없습니다.


## 15. 지금 만든 것은 무엇인가?

이번 Notebook에서 만든 것은 매우 작은 **Semantic Knowledge Base**입니다.

```text
Knowledge Documents
      ↓
BGE-M3
      ↓
Embedding Vectors
      ↓
FAISS Index
```

그리고 질문이 들어오면:

```text
Question
   ↓
BGE-M3
   ↓
Query Vector
   ↓
FAISS Search
   ↓
Top-K Knowledge
   ↓
Qwen Context
   ↓
Answer
```

이 구조는 RAG의 핵심 아이디어와 연결됩니다.

하지만 이번 실습에서는 복잡한 RAG Pipeline을 구축하는 것이 목적이 아닙니다.

목적은 다음 한 가지입니다.

> **기업의 Knowledge Asset을 검색 가능한 형태로 만들면, 범용 sLLM도 필요한 도메인 지식을 선택적으로 활용할 수 있다.**


## 16. Fine-Tuning과 연결

이제 Knowledge Asset을 활용하는 두 가지 방향을 비교할 수 있습니다.

```text
Knowledge Asset
      │
      ├─ 검색 가능한 형태로 구축
      │      ↓
      │   Embedding / FAISS
      │      ↓
      │   필요한 Knowledge를 Context로 제공
      │      ↓
      │   RAG 계열 접근
      │
      └─ 학습 데이터로 가공
             ↓
          SFT Dataset
             ↓
          PEFT / LoRA
             ↓
          Fine-Tuning
```

이번 Notebook에서는 첫 번째 방향을 **아주 작게 체험**했습니다.

이후 본 실습에서는 두 번째 방향인 **Fine-Tuning**을 중심으로 다룹니다.


## 17. 다음 실습

다음 단계에서는 PEFT(LoRA)를 적용하기 전과 후의
Trainable Parameter가 어떻게 달라지는지 확인합니다.

```text
Knowledge Asset
      ↓
BGE-M3 + FAISS Mini Knowledge Base
      ↓
PEFT / LoRA Parameter 비교
      ↓
HR SFT Dataset
      ↓
LoRA / QLoRA Fine-Tuning
      ↓
Base vs Fine-tuned Model 평가
```

이번 워밍업의 핵심 메시지는 다음과 같습니다.

> **sLLM의 차별화는 작은 모델 자체가 아니라, 기업이 가진 Knowledge Asset을 어떻게 검색하고 학습하여 활용하느냐에서 만들어질 수 있습니다.**
